# 7. Поглощение и направленность дошедшего света

Проверяем физический смысл эффекта. Необходимо различать изменение траектории
фотона и отбор траекторий, дошедших до заданного приёмника. Поглощение не
поворачивает фотоны; оно меняет веса траекторий разной длины.

In [ ]:
from pathlib import Path
import sys, time, platform
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Notebook can be started from the repo root or notebooks/course.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/lighthit').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start this notebook inside the LightHit repository')
sys.path.insert(0, str(ROOT / 'src'))
from lighthit import Medium, SolverSettings, PointGreenSolver

# Explicit public test medium. No private provider is imported.
medium = Medium(0.04, 0.05, 0.7, 1.35, 450.0, 'course-synthetic')
np.set_printoptions(precision=7, suppress=True)
print('Python:', sys.executable)
print('Platform:', platform.platform())

## 7.1. Точное тождество при однородном поглощении

Для мгновенного источника при неизменных рассеянии и скорости
$$I_{\mu_a}(\mathbf r,\mathbf s,t)=e^{-\mu_a v_gt}I_0(\mathbf r,\mathbf s,t).$$
Подставьте это в RTE: производная даёт ровно $-\mu_a I$, который сокращает
поглощение. Тождество верно по отдельности для каждого числа рассеяний.

**При фиксированном времени после испускания нормированное распределение
направлений не зависит от однородного поглощения.** Для интегрированного
сигнала длинные траектории получают меньший вес.

In [ ]:
from dataclasses import replace
from lighthit.single import single_scattering_rate
r=20.;t=r/medium.speed_m_per_ns+np.linspace(.1,300,200)
a=replace(medium,absorption_per_m=.02)
b=replace(medium,absorption_per_m=.07)
Ka=single_scattering_rate(t,r,.3,a);Kb=single_scattering_rate(t,r,.3,b)
np.testing.assert_allclose(Kb,Ka*np.exp(-(.07-.02)*medium.speed_m_per_ns*t),rtol=2e-14,atol=1e-30)
print('Exact absorption reweighting verified for the coordinate first order')

## 7.2. Что измеряем

Пусть $u=\mathbf s\cdot\widehat{\mathbf r}$ — косинус направления прихода,
$Q=\int I\,dt\,d\Omega$, $F=\int uI\,dt\,d\Omega$.
Средний радиальный косинус $\langle u\rangle=F/Q$.
Это локальная направленность относительно радиуса, не появление выделенной
мировой оси у изотропного источника.

Обозначая длину траектории $S=v_gt$, из дифференцирования нормированного
среднего получаем
$$\frac{\partial\langle u\rangle}{\partial\mu_a}=-\operatorname{Cov}(u,S).$$
Знак определяется корреляцией длины и угла. Универсального утверждения о
монотонном сужении с расстоянием из этого равенства не следует.

## 7.3. Независимый положительный Monte Carlo

Ниже траектории строятся без поглощения; вес участка равен $e^{-\mu_aS}$.
Все повторные пересечения сферического слоя учитываются. Оцениватель
интегрирует длину пути в тонком слое и делит на его объём. Это пространственное
усреднение скалярного потока, а не вероятность первого попадания.

Одни и те же траектории переиспользуются для нескольких $\mu_a$. Ошибки
оцениваются по независимым пакетам; их конечность не доказывает отсутствие
редких больших весов. Здесь нет углового спектрального решателя и кэша RTE.

In [ ]:
import importlib.util
RUN_MC = importlib.util.find_spec('numba') is not None
if RUN_MC:
    from lighthit.experimental.shell_mc import shell_estimate,ratio_with_standard_error
    t0=time.perf_counter()
    batches=shell_estimate(absorptions=[.02,.07,.14],radii_m=[20.,80.],
                           photons_per_batch=2000,batches=12,max_path_m=1000.)
    mean,se=ratio_with_standard_error(batches)
    print('MC time (possible JIT included) [s]:',time.perf_counter()-t0)
    print('absorption, radius, mean cosine >=2, standard error')
    for ia,absorption in enumerate([.02,.07,.14]):
        for ir,rad in enumerate([20.,80.]):
            print(absorption,rad,mean[ia,ir,2],se[ia,ir,2])
    fig,ax=plt.subplots()
    for ir,rad in enumerate([20.,80.]):
        ax.errorbar([.02,.07,.14],mean[:,ir,2],yerr=se[:,ir,2],marker='o',label=f'r={rad:g} m')
    ax.set(xlabel='absorption [m^-1]',ylabel='mean radial cosine, >=2 scatters');ax.legend();plt.show()
else:
    print('MC skipped: install the optional accelerate extra to execute it.')

## 7.4. Что нужно для научного вывода

Проверить размер слоя и ограничение длины траектории; увеличить число
независимых пакетов; сопоставить с другим решателем; варьировать $\mu_a$,
$\mu_s$ и форму фазовой функции. Для импульса отдельно сравнить фиксированное
время и интегрированный по времени сигнал. Следующая работа вводит
стационарный контроль, который вообще не интегрирует по вещественному $k$.

Анизотропные асимптотические световые поля изучались ранее:
Twardowski & Tonizzo, *Optics Express* 25, 18122 (2017),
[DOI 10.1364/OE.25.018122](https://doi.org/10.1364/OE.25.018122).
Новизну будущей статьи нужно устанавливать сравнением, а не самим фактом
$\langle u\rangle\ne0$.